# Covertype — pipeline **standalone** (Google Colab)

**Um único notebook** (sem pasta do TCC no Colab): `WORK_DIR`, download do Covertype (OpenML), construção do **dataset meta** (Bloco A + perturbações 10/20/30%, como no fluxograma do Diabetes adaptado ao Covertype), depois **duas saídas só com medidas** para etapas seguintes (classificador meta, etc.).

**Ordem das seções (4 a 11):** importações → PyHard (clone + teste) → H-CAT (clone + teste) → download Covertype → **Fase 0 / dataset meta** (limpeza, `problema_qualidade`, perturbações 10/20/30%) → medidas PyHard → medidas H-CAT → conferência.

**Saídas finais** (IDs + `problema_qualidade` + colunas de medidas; sem features brutas):
- `datasets/covertype_meta_pyhard_10_20_30pct.csv`
- `datasets/covertype_meta_hcat_10_20_30pct.csv`

**PyHard:** [ITA-ML/PyHard (GitLab)](https://gitlab.com/ita-ml/pyhard). A verificação importa `ClassificationMeasures`; o pipeline grande usa **lotes** (lote×N), porque N×N não cabe na RAM no Covertype completo.

**H-CAT:** `https://github.com/seedatnabeel/H-CAT`. Métodos tabulares configuráveis em `HCAT_CHARACTERIZATION_METHODS` na célula de configuração. Se o `main` quebrar, use `HCAT_REF`.

**Runtime:** GPU recomendada para H-CAT; ajuste `WORK_DIR`, `MAX_ROWS`, `PYHARD_BATCH_SIZE` na configuração.

## 1) Dependências

No **Google Colab**, evite `pip install --upgrade numpy pandas torch`: o runtime fixa versões (ex.: `pandas==2.2.2`) e atualizar gera o `ImportError` em `numpy._core.umath` e avisos de conflito com `torchaudio`.

Instalamos só o que costuma faltar: **OpenML**, deps do **H-CAT** e, para o **PyHard** clonado (sem `pip install pyhard`), **`holoviews` / `panel` / `bokeh` / `shapely`** — o `from pyhard.measures import ClassificationMeasures` puxa `base.py`, que importa esses pacotes (visualização). O cálculo em **lotes** no notebook não usa o GUI, mas o import da classe passa por esse módulo.

**Se a sessão já ficou quebrada:** *Runtime → Restart session*, depois rode do início.

In [ ]:
# Colab já traz numpy, pandas, scipy, scikit-learn e torch (CUDA) compatíveis com google-colab.
# NÃO use --upgrade nesses pacotes: quebra imports (ex.: numpy._core.umath) e torch vs torchaudio.
!pip install -q openml tqdm
# PyHard (git clone): ClassificationMeasures → pyhard.base → holoviews/panel/bokeh/shapely
!pip install -q holoviews panel bokeh shapely
!pip install -q cleanlab aum augly opencv-python-headless Pillow

## 2) Configuração — **único lugar obrigatório para editar**

In [ ]:
from pathlib import Path
import os

# Diretório de trabalho (Colab: /content/...). Nada depende do repositório TCC.
WORK_DIR = Path("/content/covertype_meta_pipeline")
RAW_DIR = WORK_DIR / "raw"
OUT_DIR = WORK_DIR / "datasets"
PYHARD_REPO_DIR = WORK_DIR / "ita_pyhard"  # git clone ITA-ML PyHard
HCAT_DIR = WORK_DIR / "H-CAT"

SEED = 42
OPENML_DATASET_ID = 1596
USE_OPENML_DOWNLOAD = True  # False se você já tiver colocado covertype_full.csv em RAW_DIR

# None = usar **todo** o Covertype após dropna(y) (dataset completo; adequado a máquinas com muita RAM, ex. 50+ GB).
# Inteiro = subamostra estratificada só se você quiser testar mais rápido.
MAX_ROWS = None

PYHARD_GIT_URL = "https://gitlab.com/ita-ml/pyhard.git"
PYHARD_GIT_REF = None  # ex. hash de commit; None = branch padrão (clone raso)

# PyHard em lotes: menor batch => menos pico de RAM (com N grande ainda é vários GB por lote)
PYHARD_BATCH_SIZE = 256
COMPUTE_F1_OVERLAP = False  # True é muito lento no Covertype (7 classes)

# H-CAT
HCAT_GIT_URL = "https://github.com/seedatnabeel/H-CAT.git"
HCAT_REF = None  # ex.: "abc1234" para checkout em commit específico; None = branch padrão
HCAT_EPOCHS = 5
HCAT_BATCH_SIZE = 256

# Colunas extras a remover após o CSV bruto (OpenML costuma vir limpo; deixe vazio se não precisar).
DROP_COVTYPE_COLS = []

# Métodos H-CAT usados para dados **tabulares** neste notebook (alinhado ao fluxograma / TCC).
HCAT_CHARACTERIZATION_METHODS = [
    "aum",
    "data_uncert",
    "el2n",
    "grand",
    "cleanlab",
    "forgetting",
    "vog",
    "prototypicality",
    "loss",
    "conf_agree",
]

# Nomes das medidas PyHard gravadas no CSV meta (N1 / Usefulness ficam NaN neste pipeline em lotes).
PYHARD_META_MEASURES = [
    "kDN", "DCP", "TD_P", "TD_U", "CL", "CLD", "N1", "N2", "LSC", "LSR",
    "Harmfulness", "Usefulness", "F1",
]

for d in (WORK_DIR, RAW_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.chdir(WORK_DIR)
print("WORK_DIR:", WORK_DIR.resolve())

## 3) Montar Google Drive (opcional)

Use se quiser persistir `WORK_DIR` no Drive (altere `WORK_DIR` acima para `/content/drive/MyDrive/.../covertype_meta_pipeline`).

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fora do Colab — ignorando Drive.")

## 4) Importações (bibliotecas padrão e stack ML)

Execute **após** configuração (e Drive, se usar). Todas as importações usadas nas seções seguintes.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from itertools import combinations
from pathlib import Path

import numpy as np
import openml
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import tree
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

print("NumPy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__)

## 5) PyHard — clone do repositório e verificação

Clona o [ITA-ML/PyHard](https://gitlab.com/ita-ml/pyhard) e confirma que o pacote importa (incl. `ClassificationMeasures`). O cálculo em escala real está na seção de medidas em **lotes**.

In [ ]:
if not (PYHARD_REPO_DIR / "pyhard").is_dir():
    if PYHARD_GIT_REF:
        subprocess.run(["git", "clone", PYHARD_GIT_URL, str(PYHARD_REPO_DIR)], check=True)
        subprocess.run(
            ["git", "-C", str(PYHARD_REPO_DIR), "checkout", PYHARD_GIT_REF],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", PYHARD_GIT_URL, str(PYHARD_REPO_DIR)],
            check=True,
        )
else:
    print("PyHard (ITA-ML) já clonado em", PYHARD_REPO_DIR)

_p = str(PYHARD_REPO_DIR.resolve())
if _p not in sys.path:
    sys.path.insert(0, _p)
print("sys.path[0] (PyHard):", _p)

In [ ]:
from pyhard.measures import ClassificationMeasures

print("PyHard OK — ClassificationMeasures:", getattr(ClassificationMeasures, "__name__", ClassificationMeasures))
print("Medidas planejadas no CSV meta (lotes):", PYHARD_META_MEASURES)

## 6) H-CAT — clone do repositório e verificação

Clona o repositório oficial e importa módulos usados para dados **tabulares**. Os métodos efetivos do treino vêm de `HCAT_CHARACTERIZATION_METHODS` na configuração.

In [ ]:
if not (HCAT_DIR / "src").is_dir():
    if HCAT_REF:
        subprocess.run(["git", "clone", HCAT_GIT_URL, str(HCAT_DIR)], check=True)
        subprocess.run(["git", "-C", str(HCAT_DIR), "checkout", HCAT_REF], check=True)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", HCAT_GIT_URL, str(HCAT_DIR)],
            check=True,
        )
else:
    print("H-CAT já existe em", HCAT_DIR)

_h = str(HCAT_DIR.resolve())
if _h not in sys.path:
    sys.path.insert(0, _h)
print("sys.path (H-CAT):", _h)

In [ ]:
from src.dataloader import MultiFormatDataLoader
from src.models import MLP
from src.trainer import PyTorchTrainer
from src.utils import seed_everything

print("H-CAT OK — imports de src.* (tabular).")
print("Métodos configurados:", HCAT_CHARACTERIZATION_METHODS)
print("Device disponível (pré-treino):", "cuda" if torch.cuda.is_available() else "cpu")

## 7) Dataset Covertype — carregamento (`raw/covertype_full.csv`)

OpenML (`OPENML_DATASET_ID`) ou CSV local com `USE_OPENML_DOWNLOAD = False`. No fetch: `dropna` em `y`, `LabelEncoder` (0..K-1). Pré-processamento adicional (colunas, NaN em features, Fase 0 / meta) na **seção 8**.

In [ ]:
def fetch_covertype_openml(output_csv: Path, dataset_id: int = 1596) -> Path:
    print(f"OpenML dataset {dataset_id}...")
    dataset = openml.datasets.get_dataset(dataset_id)
    X, y, _, attribute_names = dataset.get_data(
        dataset_format="dataframe",
        target=dataset.default_target_attribute,
    )
    df = pd.DataFrame(X, columns=attribute_names)
    df["y"] = y
    df = df.dropna()
    le = LabelEncoder()
    df["y"] = le.fit_transform(df["y"].astype(str))
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Salvo {output_csv} | linhas={len(df):,} | classes={sorted(df['y'].unique().tolist())}")
    return output_csv

COVER_CSV = RAW_DIR / "covertype_full.csv"
if USE_OPENML_DOWNLOAD:
    fetch_covertype_openml(COVER_CSV, OPENML_DATASET_ID)
else:
    assert COVER_CSV.is_file(), (
        f"Coloque covertype_full.csv em {COVER_CSV} ou defina USE_OPENML_DOWNLOAD = True"
    )
    print("Usando CSV existente:", COVER_CSV)

## 8) Dataset meta — pré-processamento e Fase 0 (fluxograma TCC)

Nesta seção única: lê `COVER_CSV`, aplica `DROP_COVTYPE_COLS`, `dropna` em todas as colunas, garante `instance_id`, grava `covertype_original.csv` (com `y` verdadeiro), monta **Bloco A** (sem coluna `y` no CSV; `problema_qualidade=nao`, `origem=A`), **Blocos B** 10/20/30% com *mislabeling* uniforme em `y` (`problema_qualidade=sim`, `origem=B`), e concatena **A + B₁₀ + B₂₀ + B₃₀** → `covertype_conjunto_10_20_30pct.csv` (mesma lógica do diabetes no doc do fluxograma).


In [ ]:
ID_COL = "instance_id"
TARGET_COL = "y"
PROPORTIONS = [0.1, 0.2, 0.3]

def uniform_mislabeling(labels, p, seed=None):
    if seed is not None:
        np.random.seed(seed)
    if hasattr(labels, "values"):
        labels = labels.values
    labels = np.asarray(labels, dtype=int)
    n = len(labels)
    n_flip = int(round(p * n))
    flip_indices = np.random.choice(n, size=n_flip, replace=False)
    unique = np.unique(labels)
    n_classes = len(unique)
    label_to_idx = {lb: i for i, lb in enumerate(unique)}
    idx_to_label = {i: lb for lb, i in label_to_idx.items()}
    flipped = labels.copy()
    for idx in flip_indices:
        orig_idx = label_to_idx[int(labels[idx])]
        choices = [i for i in range(n_classes) if i != orig_idx]
        new_idx = int(np.random.choice(choices))
        flipped[idx] = idx_to_label[new_idx]
    return flip_indices, flipped

def run_phase0(input_csv: Path, out_dir: Path, max_rows: int | None, seed: int = 42):
    df = pd.read_csv(input_csv)
    if TARGET_COL not in df.columns:
        raise ValueError(f"Falta coluna {TARGET_COL}")
    drop_cols = [c for c in DROP_COVTYPE_COLS if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
        print("Removidas colunas (config):", drop_cols)
    df = df.dropna().reset_index(drop=True)
    df[TARGET_COL] = df[TARGET_COL].astype(int)
    if ID_COL not in df.columns:
        df.insert(0, ID_COL, np.arange(len(df), dtype=np.int64))
    else:
        df[ID_COL] = df[ID_COL].astype(np.int64)
    labels = df[TARGET_COL].values
    print(f"Linhas após dropna(y): {len(df)}")

    if max_rows is not None and len(df) > max_rows:
        idx, _ = train_test_split(
            df.index, train_size=max_rows, stratify=df[TARGET_COL], random_state=seed
        )
        df = df.loc[idx].reset_index(drop=True)
        df[ID_COL] = np.arange(len(df), dtype=np.int64)
        labels = df[TARGET_COL].values
        print(f"Subamostra estratificada: {len(df)} (max_rows={max_rows})")

    out_dir.mkdir(parents=True, exist_ok=True)
    path_original = out_dir / "covertype_original.csv"
    df.to_csv(path_original, index=False)
    print("Salvo:", path_original)

    features_cols = [c for c in df.columns if c != TARGET_COL]
    block_a = df[features_cols].copy()
    block_a["problema_qualidade"] = "nao"
    block_a["origem"] = "A"
    block_a.to_csv(out_dir / "covertype_block_a.csv", index=False)

    for p in PROPORTIONS:
        np.random.seed(seed)
        flag_ids, flipped = uniform_mislabeling(labels, p, seed=seed)
        df_p = df.copy()
        df_p[TARGET_COL] = flipped
        block_b = df_p.loc[flag_ids, features_cols].copy()
        block_b["problema_qualidade"] = "sim"
        block_b["origem"] = "B"
        suf = f"{int(p * 100)}pct"
        block_b.to_csv(out_dir / f"covertype_block_b_{suf}.csv", index=False)
        conj = pd.concat([block_a, block_b], ignore_index=True)
        conj.to_csv(out_dir / f"covertype_conjunto_{suf}.csv", index=False)
        print(f"conjunto_{suf}: {len(conj)} linhas")

    p10 = out_dir / "covertype_conjunto_10pct.csv"
    if p10.is_file():
        shutil.copy(p10, out_dir / "covertype_conjunto.csv")

    blocks_b = [pd.read_csv(out_dir / f"covertype_block_b_{int(p * 100)}pct.csv") for p in PROPORTIONS]
    conj_full = pd.concat([block_a] + blocks_b, ignore_index=True)
    path_full = out_dir / "covertype_conjunto_10_20_30pct.csv"
    conj_full.to_csv(path_full, index=False)
    print("Salvo:", path_full, "| linhas:", len(conj_full))
    return path_full, path_original

CONJUNTO_CSV, ORIGINAL_CSV = run_phase0(COVER_CSV, OUT_DIR, MAX_ROWS, SEED)
print("CONJUNTO_CSV:", CONJUNTO_CSV)
print("ORIGINAL_CSV:", ORIGINAL_CSV)

## 9) Medidas PyHard (lotes) → `covertype_meta_pyhard_10_20_30pct.csv`

Usa `covertype_conjunto_10_20_30pct.csv` + merge de `y` via `covertype_original.csv`. Gower + kDN, N2, LSC, LSR, Harmfulness em **lotes**; DCP, TD_*, CL, CLD via árvores/NB. Colunas do CSV meta: `instance_id`, `problema_qualidade` e `PYHARD_META_MEASURES` da config.


In [ ]:
EXCLUDE_FROM_FEATURES = {ID_COL, "origem", "problema_qualidade"}
K_KDN = 10

def gower_distance_query_ref(X_query: np.ndarray, X_ref: np.ndarray) -> np.ndarray:
    nq, n_feat = X_query.shape
    nr = X_ref.shape[0]
    out = np.zeros((nq, nr), dtype=np.float64)
    for j in range(n_feat):
        qj = X_query[:, j : j + 1]
        rj = X_ref[:, j : j + 1]
        rng = max(np.ptp(rj), 1e-8)
        out += pairwise_distances(qj, rj, metric="manhattan") / rng
    return out / n_feat

def measures_from_dist_batch(dist_batch, y_full, y_batch, k=10):
    n_batch, N = dist_batch.shape
    idx_sorted = np.argsort(dist_batch, axis=1)
    y_full_flat = np.asarray(y_full).ravel()
    kdn = np.zeros(n_batch)
    for i in range(n_batch):
        ind = idx_sorted[i, : k + 1]
        yi = y_batch[i]
        neigh_labels = y_full_flat[ind]
        kdn[i] = np.sum(neigh_labels[1:] != yi) / k
    n2 = np.zeros(n_batch)
    lsc = np.zeros(n_batch)
    lsr = np.zeros(n_batch)
    n_class = {c: np.sum(y_full_flat == c) for c in np.unique(y_full_flat)}
    for i in range(n_batch):
        ind = idx_sorted[i]
        dist_i = dist_batch[i]
        d_sorted = np.sort(dist_i)
        labels_sorted = y_full_flat[ind]
        yi = y_batch[i]
        same = labels_sorted == yi
        diff = ~same
        idx_first_same = np.where(same)[0]
        idx_first_diff = np.where(diff)[0]
        if len(idx_first_same) < 2 or len(idx_first_diff) == 0:
            n2[i] = np.nan
            lsc[i] = np.nan
            lsr[i] = np.nan
            continue
        dist_same_1 = d_sorted[idx_first_same[1]]
        dist_ne = d_sorted[idx_first_diff[0]]
        n2[i] = dist_same_1 / max(dist_ne, 1e-15)
        ls_size = np.argmax(diff)
        lsc[i] = 1 - ls_size / max(n_class.get(yi, 1), 1)
        idx_last_same = len(labels_sorted) - 1 - np.argmax(same[::-1])
        denom = d_sorted[idx_last_same]
        lsr[i] = 1 - min(1.0, dist_ne / max(denom, 1e-15))
    n2 = 1 - 1 / (n2 + 1)
    return {"kDN": kdn, "N2": n2, "LSC": lsc, "LSR": lsr}

def harmfulness_from_ne(ne_indices, y_full):
    N = len(ne_indices)
    n_class = pd.Series(y_full).value_counts()
    denom = np.array([N - n_class.get(y_full[i], 0) for i in range(N)], dtype=float)
    denom = np.where(denom <= 0, 1, denom)
    return np.bincount(ne_indices, minlength=N) / denom

def run_pyhard(conjunto_path: Path, original_path: Path, output_path: Path, batch_size: int):
    conjunto = pd.read_csv(conjunto_path)
    original = pd.read_csv(original_path)
    id_to_y = original.set_index(ID_COL)["y"]
    conjunto = conjunto.copy()
    conjunto["y"] = conjunto[ID_COL].map(id_to_y)
    conjunto = conjunto.dropna(subset=["y"]).copy()
    conjunto["y"] = conjunto["y"].astype(int)

    feature_cols = [
        c for c in conjunto.columns if c not in EXCLUDE_FROM_FEATURES and c != TARGET_COL
    ]
    X_all = conjunto[feature_cols].copy()
    for col in X_all.columns:
        if X_all[col].dtype.name == "category":
            X_all[col] = X_all[col].cat.codes
        elif not pd.api.types.is_numeric_dtype(X_all[col]):
            X_all[col] = pd.Categorical(X_all[col].astype(str)).codes
    valid = X_all.notna().all(axis=1)
    X_all = X_all.loc[valid].to_numpy().astype(np.float64)
    y_all = conjunto.loc[valid, "y"].values.astype(int)
    problema = conjunto.loc[valid, "problema_qualidade"].values
    ids = conjunto.loc[valid, ID_COL].values

    N, F = X_all.shape
    bs = int(batch_size)
    print(f"PyHard N={N}, F={F}, batch_size={bs}")
    print(f"Pico aprox. distâncias (GB): {bs * N * 8 / 1e9:.2f}")

    print("Ajustando árvores e Naive Bayes...")
    t0 = time.time()
    dtc = tree.DecisionTreeClassifier(min_samples_split=2, criterion="gini", random_state=SEED)
    dtc.fit(X_all, y_all)
    params = {"ccp_alpha": np.linspace(0.001, 0.1, num=20)}
    dtc_pruned = tree.DecisionTreeClassifier(criterion="gini", random_state=SEED)
    clf = GridSearchCV(dtc_pruned, params, n_jobs=-1)
    clf.fit(X_all, y_all)
    dtc_pruned = tree.DecisionTreeClassifier(
        criterion="gini", ccp_alpha=clf.best_params_["ccp_alpha"], random_state=SEED
    )
    dtc_pruned.fit(X_all, y_all)
    n_c = len(np.unique(y_all))
    nb = GaussianNB(priors=np.ones(n_c) / n_c)
    calibrated_nb = CalibratedClassifierCV(
        estimator=nb, method="sigmoid", cv=3, ensemble=False, n_jobs=-1
    )
    calibrated_nb.fit(X_all, y_all)
    print(f"Modelos base em {time.time() - t0:.1f}s")

    leaf_id = dtc_pruned.apply(X_all)
    leaf_counts = pd.Series(leaf_id).value_counts()
    same_class_count = np.zeros(N)
    for i in range(N):
        lid = leaf_id[i]
        mask = leaf_id == lid
        same_class_count[i] = np.sum(y_all[mask] == y_all[i])
    dcp = 1 - (same_class_count / np.array([leaf_counts[l] for l in leaf_id]))

    path_u = dtc.decision_path(X_all)
    path_p = dtc_pruned.decision_path(X_all)
    td_u = (np.asarray(path_u.sum(axis=1)).ravel() - 1).astype(float) / max(dtc.get_depth(), 1)
    td_p = (np.asarray(path_p.sum(axis=1)).ravel() - 1).astype(float) / max(dtc_pruned.get_depth(), 1)

    proba = calibrated_nb.predict_proba(X_all)
    classes_nb = calibrated_nb.classes_
    cl_idx = np.argmax(y_all.reshape(-1, 1) == classes_nb, axis=1)
    CL = 1 - proba[np.arange(N), cl_idx]
    CLD_row = proba[np.arange(N), cl_idx] - np.max(
        np.where(np.arange(proba.shape[1])[None, :] != cl_idx[:, None], proba, -np.inf),
        axis=1,
    )
    CLD = (1 - CLD_row) / 2

    if COMPUTE_F1_OVERLAP:
        classes = np.unique(y_all)
        pairs = list(combinations(classes, 2))
        F1_arr = np.zeros(N)
        for c1, c2 in pairs:
            mask = (y_all == c1) | (y_all == c2)
            sub_y = y_all[mask]
            sub_x = X_all[mask]
            for j in range(F):
                f_c1 = sub_x[sub_y == c1, j]
                f_c2 = sub_x[sub_y == c2, j]
                if len(f_c1) == 0 or len(f_c2) == 0:
                    continue
                maxmin_j = max(np.min(f_c1), np.min(f_c2))
                minmax_j = min(np.max(f_c1), np.max(f_c2))
                in_overlap = (X_all[:, j] >= maxmin_j) & (X_all[:, j] <= minmax_j)
                F1_arr += in_overlap.astype(float) / F
        F1_arr /= max(len(pairs), 1)
    else:
        F1_arr = np.full(N, np.nan)

    ne_indices = np.zeros(N, dtype=int)
    n_batches = (N + bs - 1) // bs
    print(f"Lotes Gower: {n_batches}")
    t_start = time.time()
    list_kdn, list_n2, list_lsc, list_lsr = [], [], [], []
    for b in range(n_batches):
        start = b * bs
        end = min(start + bs, N)
        X_batch = X_all[start:end]
        dist_batch = gower_distance_query_ref(X_batch, X_all)
        for i in range(end - start):
            dist_batch[i, start + i] = np.inf
        y_batch = y_all[start:end]
        res = measures_from_dist_batch(dist_batch, y_all, y_batch, k=K_KDN)
        list_kdn.append(res["kDN"])
        list_n2.append(res["N2"])
        list_lsc.append(res["LSC"])
        list_lsr.append(res["LSR"])
        idx_sorted = np.argsort(dist_batch, axis=1)
        for i in range(end - start):
            labels_sorted = y_all[idx_sorted[i]]
            yi = y_all[start + i]
            diff_pos = np.where(labels_sorted != yi)[0]
            if len(diff_pos) > 0:
                ne_indices[start + i] = idx_sorted[i, diff_pos[0]]
        if (b + 1) % 5 == 0 or b == n_batches - 1:
            print(f"  lote {b + 1}/{n_batches} ({time.time() - t_start:.1f}s)")

    kdn_all = np.concatenate(list_kdn)
    n2_all = np.concatenate(list_n2)
    lsc_all = np.concatenate(list_lsc)
    lsr_all = np.concatenate(list_lsr)
    harm = harmfulness_from_ne(ne_indices, y_all)
    meta = pd.DataFrame(
        {
            ID_COL: ids,
            "problema_qualidade": problema,
            "kDN": kdn_all,
            "DCP": dcp,
            "TD_P": td_p,
            "TD_U": td_u,
            "CL": CL,
            "CLD": CLD,
            "N1": np.full(N, np.nan),
            "N2": n2_all,
            "LSC": lsc_all,
            "LSR": lsr_all,
            "Harmfulness": harm,
            "Usefulness": np.full(N, np.nan),
            "F1": F1_arr,
        }
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    meta.to_csv(output_path, index=False)
    print("Salvo:", output_path, meta.shape)

META_PYHARD = OUT_DIR / "covertype_meta_pyhard_10_20_30pct.csv"
run_pyhard(CONJUNTO_CSV, ORIGINAL_CSV, META_PYHARD, PYHARD_BATCH_SIZE)

## 10) Medidas H-CAT → `covertype_meta_hcat_10_20_30pct.csv`

Mesmo conjunto; `HCAT_CHARACTERIZATION_METHODS` na config. Saída: `instance_id`, `problema_qualidade` e uma coluna por método.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from src.dataloader import MultiFormatDataLoader
from src.models import MLP
from src.trainer import PyTorchTrainer
from src.utils import seed_everything

EXCLUDE_HCAT = {ID_COL, "origem", "problema_qualidade"}

def load_hcat_xy(conjunto_path: Path, original_path: Path):
    conjunto = pd.read_csv(conjunto_path)
    original = pd.read_csv(original_path)
    orig_sub = original[[ID_COL, "y"]].drop_duplicates()
    conjunto = conjunto.merge(orig_sub, on=ID_COL, how="left")
    conjunto = conjunto.dropna(subset=["y"]).copy()
    conjunto["y"] = conjunto["y"].astype(int)
    feature_cols = [c for c in conjunto.columns if c not in EXCLUDE_HCAT and c != "y"]
    df_work = conjunto[feature_cols + ["y"]].copy()
    for col in df_work.columns:
        if col == "y":
            continue
        if df_work[col].dtype.name == "category":
            df_work[col] = df_work[col].cat.codes
        elif not pd.api.types.is_numeric_dtype(df_work[col]):
            df_work[col] = pd.Categorical(df_work[col].astype(str)).codes
    valid_idx = df_work.dropna().index
    df_work = df_work.loc[valid_idx].reset_index(drop=True)
    conjunto_valid = conjunto.loc[valid_idx].reset_index(drop=True)
    df_work["y"] = df_work["y"].astype(int)
    X = df_work[feature_cols].to_numpy().astype(np.float32)
    y = df_work["y"].values
    return X, y, conjunto_valid

def _to_flat_array(s):
    if isinstance(s, (list, tuple)):
        if len(s) == 0:
            return np.array([])
        if hasattr(s[0], "shape"):
            return np.concatenate([np.asarray(x).ravel() for x in s])
        return np.asarray(s).ravel()
    return np.asarray(s).ravel()

def extract_raw_scores(hardness_dict):
    raw_scores_dict = {}
    for method in hardness_dict.keys():
        obj = hardness_dict[method]
        if method in ("dataiq", "datamaps"):
            raw = obj.compute_scores(datamaps=(method == "datamaps"))
            _, confidence = raw
            raw_scores = _to_flat_array(confidence)
        elif method == "cleanlab":
            raw_scores, _ = obj.compute_scores()
            raw_scores = _to_flat_array(raw_scores)
        else:
            if getattr(obj, "_scores", None) is None:
                obj.compute_scores()
            raw_scores = _to_flat_array(obj.scores)
        if np.isnan(raw_scores).any():
            raw_scores = np.nan_to_num(raw_scores, nan=np.nanmean(raw_scores))
        raw_scores_dict[method] = raw_scores
    return raw_scores_dict

def run_hcat(conjunto_path: Path, original_path: Path, output_path: Path):
    seed_everything(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    X, y, conjunto_valid = load_hcat_xy(conjunto_path, original_path)
    N, n_features = X.shape
    n_classes = len(np.unique(y))
    print(f"N={N}, features={n_features}, classes={n_classes}")

    dataloader_class = MultiFormatDataLoader(
        data=(X, y),
        target_column=None,
        data_type="numpy",
        data_modality="tabular",
        batch_size=HCAT_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        transform=None,
        image_transform=None,
        perturbation_method="uniform",
        p=0.0,
        rule_matrix=None,
        atypical_marginal=[],
    )
    dataloader, dataloader_unshuffled = dataloader_class.get_dataloader()

    model = MLP(input_size=n_features, nlabels=n_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    trainer = PyTorchTrainer(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        lr=0.001,
        epochs=HCAT_EPOCHS,
        total_samples=N,
        num_classes=n_classes,
        device=device,
        characterization_methods=HCAT_CHARACTERIZATION_METHODS,
    )
    print("Treinando H-CAT...")
    trainer.fit(dataloader, dataloader_unshuffled)
    hardness_dict = trainer.get_hardness_methods()
    print("Extraindo medidas...")
    raw_scores_dict = extract_raw_scores(hardness_dict)
    df_measures = pd.DataFrame(raw_scores_dict)
    meta = pd.concat(
        [
            conjunto_valid[[ID_COL, "problema_qualidade"]].reset_index(drop=True),
            df_measures.reset_index(drop=True),
        ],
        axis=1,
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    meta.to_csv(output_path, index=False)
    print("Salvo:", output_path, meta.shape)

META_HCAT = OUT_DIR / "covertype_meta_hcat_10_20_30pct.csv"
run_hcat(CONJUNTO_CSV, ORIGINAL_CSV, META_HCAT)

## 11) Conferência e download

Dois CSVs **só com identificação + rótulo de qualidade + medidas** (sem features brutas). No Colab: *Files* → `covertype_meta_pipeline/datasets/`.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

for p in (META_PYHARD, META_HCAT):
    df = pd.read_csv(p, nrows=3)
    print(p.name, list(df.columns)[:8], "...")
    display(df)